# <center>Finite Element Method: 3D NS equation</center>

# 1. Import necessary packages

In [1]:
import numpy as np
from dolfin import *
from meshio import write
import meshio
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import gmsh
import sys, time, os
from PIL import Image
from scipy.sparse.linalg import eigs
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import inv
from petsc4py import PETSc
import time
# plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

def process_bar(num, total):
    rank = MPI.rank(MPI.comm_world)  # Obtian the rank of the process
    if rank == 0:  # Only the main process runs this
        rate = float(num)/total
        ratenum = 100*rate
        rsr = '\r[{}{}]{:.2f}%'.format('*'*int(ratenum),' '*(100-int(ratenum)), ratenum)
        sys.stdout.write(rsr)
        sys.stdout.flush()

comm = MPI.comm_world
rank = MPI.rank(comm)
set_log_level(LogLevel.ERROR)

/home/kazeyoubi/miniconda3/envs/fenics/lib/python3.11/site-packages/ufl/__init__.py:250: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# 2. Create mesh

In [2]:
# Define rectangular domain (now 3D box)
D = 0.1
L = 21*D   # length in x
H = 14*D   # height in y
W = 4*D   # width in z (thickness)

# Define cylinder
xc = 5*D
yc = 7*D
rc = D/2

mesh_size = 0.1
mesh_size_fine = 0.05
mesh_size_coarse = mesh_size
R_inner = 0.2
R_outer = 0.4

gmsh.initialize()
gmsh.model.add("cylinder_3d")

# Create 3D box
box = gmsh.model.occ.addBox(0, 0, 0, L, H, W)

# Create cylinder (along Z-axis)
cylinder = gmsh.model.occ.addCylinder(xc, yc, 0, 0, 0, W, rc)

# Cut cylinder from box
gmsh.model.occ.cut([(3, box)], [(3, cylinder)])
gmsh.model.occ.synchronize()

gmsh.option.setNumber("Mesh.MeshSizeMax", mesh_size)

# --- Set to use first-order (linear) elements ---
gmsh.option.setNumber("Mesh.ElementOrder", 1)

surfaces = gmsh.model.getEntities(dim=2)
boundary_surfaces = {}

tol = 1e-6

for dim, tag in surfaces:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    xmin, ymin, zmin, xmax, ymax, zmax = bbox

    if abs(xmin) < tol and abs(xmax) < tol:
        name = "left"
    elif abs(xmin - L) < tol and abs(xmax - L) < tol:
        name = "right"
    elif abs(ymin) < tol and abs(ymax) < tol:
        name = "bottom"
    elif abs(ymin - H) < tol and abs(ymax - H) < tol:
        name = "top"
    elif abs(zmin) < tol and abs(zmax) < tol:
        name = "front"
    elif abs(zmin - W) < tol and abs(zmax - W) < tol:
        name = "back"
    else:
        name = "cylinder"  # remaining is cylinder wall

    boundary_surfaces[name] = tag

# Add physical groups for boundaries
boundnames = {"left": 1, "right": 2, "bottom": 3, "top": 4, "front": 5, "back": 6, "cylinder": 7}
for name, tag in boundary_surfaces.items():
    physical_tag = boundnames[name]
    gmsh.model.addPhysicalGroup(2, [tag], physical_tag)
    gmsh.model.setPhysicalName(2, physical_tag, name)

# Add physical volume
volume_entities = gmsh.model.getEntities(dim=3)
gmsh.model.addPhysicalGroup(3, [ve[1] for ve in volume_entities], tag=10)
gmsh.model.setPhysicalName(3, 10, "fluid")

# Distance field from cylinder surface
distance_field = gmsh.model.mesh.field.add("Distance")
gmsh.model.mesh.field.setNumbers(distance_field, "FacesList", [boundary_surfaces["cylinder"]])

threshold_field = gmsh.model.mesh.field.add("Threshold")
gmsh.model.mesh.field.setNumber(threshold_field, "IField", distance_field)
gmsh.model.mesh.field.setNumber(threshold_field, "LcMin", mesh_size_fine)
gmsh.model.mesh.field.setNumber(threshold_field, "LcMax", mesh_size_coarse)
gmsh.model.mesh.field.setNumber(threshold_field, "DistMin", R_inner)
gmsh.model.mesh.field.setNumber(threshold_field, "DistMax", R_outer)

gmsh.model.mesh.field.setAsBackgroundMesh(threshold_field)

# Generate 3D mesh
gmsh.model.mesh.generate(3)

gmsh.write("cylinder_3d.msh")

gmsh.finalize()

Info    : Meshing 1D...                                                                                                      
Info    : [  0%] Meshing curve 13 (Circle)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 20%] Meshing curve 15 (Circle)
Info    : [ 30%] Meshing curve 16 (Line)
Info    : [ 30%] Meshing curve 17 (Line)
Info    : [ 40%] Meshing curve 18 (Line)
Info    : [ 50%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 70%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 80%] Meshing curve 24 (Line)
Info    : [ 90%] Meshing curve 25 (Line)
Info    : [ 90%] Meshing curve 26 (Line)
Info    : [100%] Meshing curve 27 (Line)
Info    : Done meshing 1D (Wall 0.00281193s, CPU 0.001998s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 7 (Cylinder, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 8 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 9 (Plane, Fro

In [1]:
msh = meshio.gmsh.read("cylinder_3d.msh")

meshio.write(
    "mesh.xml",
    meshio.Mesh(
        points=msh.points,
        cells=msh.cells,
        cell_data=msh.cell_data,
        field_data=msh.field_data,
        point_data=msh.point_data
    )
)

mesh = Mesh()
File("mesh.xml") >> mesh

print("Load mesh successfully!")
print("Number of vertices:", mesh.num_vertices())
print("Number of cells:", mesh.num_cells())

NameError: name 'meshio' is not defined

In [4]:
mesh

# 3. Define function space

In [5]:
# 3D vector space
V = VectorFunctionSpace(mesh, "P", 2)  # P2-P1 Taylor-Hood
Q = FunctionSpace(mesh, "P", 1)

# 4. Set boundary conditions

In [6]:
class LeftBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[0], 0.0) and on_boundary

class RightBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[0], L) and on_boundary

class BottomBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[1], 0.0) and on_boundary

class TopBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[1], H) and on_boundary

class FrontBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[2], 0.0) and on_boundary

class BackBound(SubDomain):
    def inside(self, x, on_boundary):
        return near(x[2], W) and on_boundary

class CylinderBound(SubDomain):
    def inside(self, x, on_boundary):
        dist = sqrt((x[0] - xc)**2 + (x[1] - yc)**2)
        return on_boundary and near(dist, rc, 1e-2)

left = LeftBound()
right = RightBound()
top = TopBound()
bottom = BottomBound()
front = FrontBound()
back = BackBound()
cylinder = CylinderBound()

boundaries = MeshFunction("size_t", mesh, 2)
boundaries.set_all(0)

left.mark(boundaries, 1)
right.mark(boundaries, 2)
top.mark(boundaries, 3)
bottom.mark(boundaries, 4)
front.mark(boundaries, 5)
back.mark(boundaries, 6)
cylinder.mark(boundaries, 7)

ds = Measure('ds', domain=mesh, subdomain_data=boundaries)
dS = Measure('dS', domain=mesh)

dx = dx(metadata={'quadrature_degree': 4})
ds = ds(metadata={'quadrature_degree': 4})

# 5. Define trial and test function

In [7]:
u = TrialFunction(V)
v = TestFunction(V)
p = TrialFunction(Q)
q = TestFunction(Q)

u_n = Function(V)
p_n = Function(Q)
u1 = Function(V)
p1 = Function(Q)

# 6. Define weak equation

In [8]:
# Parameter
v0 = 10000      # Inflow velocity
nu = 0     # Viscosity
dt = 0.5 * mesh.hmin()  # Time step
T = 1.0       # End of time
num_steps = 150

# Inflow velocity
# uin = Expression(('v0', '0.0', '0.0'), v0=v0, degree=2)

np.random.seed(42)  # Random seed
random_field = np.random.rand(100) * 0.002 - 0.001  # sample random number in [-0.001, 0.001]

# Interpolate random perturbation
def random_perturbation(x):
    return random_field[int((x[2]/W)*100)]

# Inflow velocity with random perturbation
class RandomInflow(UserExpression):
    def __init__(self, v0, epsilon, W, **kwargs):
        super().__init__(**kwargs)
        self.v0 = v0
        self.epsilon = epsilon
        self.W = W
    
    def eval(self, values, x):
        perturb = self.epsilon # * np.sin(2*np.pi*x[2]/self.W)
        values[0] = self.v0 + self.v0*perturb
        values[1] = 0.0
        values[2] = 0.0
    
    def value_shape(self):
        return (3,)

uin = RandomInflow(v0=v0, epsilon=0.001, W=W, degree=2)

# Initial fields
u0 = interpolate(uin, V)
p0 = interpolate(Constant(0.0), Q)

u_n.assign(u0)
p_n.assign(p0)

# Boundary conditions
bcu = [
    DirichletBC(V, Constant((v0, 0, 0)), boundaries, 1),      # left, inflow
#    DirichletBC(V, Constant((0, 0, 0)), boundaries, 2),  # right, outflow
    DirichletBC(V, Constant((0, 0, 0)), boundaries, 3),  # top
    DirichletBC(V, Constant((0, 0, 0)), boundaries, 4),  # bottom
    DirichletBC(V, Constant((0, 0, 0)), boundaries, 5),  # front
    DirichletBC(V, Constant((0, 0, 0)), boundaries, 6),  # back
#    DirichletBC(V, Constant((0, 0, 0)), boundaries, 7),  # cylinder
]

bcp = [
    DirichletBC(Q, Constant(0.0), boundaries, 2),
]

bcs = [bcu, bcp]

In [9]:
# --- Nitsche slip term on cylinder boundary ds(7) ---

def sigma(u, p):
    return -p*Identity(3) + nu*(grad(u) + grad(u).T)  # Stress tensor

theta = 0.5  # Crank-Nicolson
u_mid = theta*u + (1 - theta)*u_n
u_mid1 = theta*u1 + (1 - theta)*u_n
n = FacetNormal(mesh)
h = CellDiameter(mesh)

beta = 0           # Navier slip coefficiet

# Nitsche punishment parameter
alpha = Constant(10.0)   # Large alpha gets close to "no penetration"

# 1. Traditional boundary term
nitsche_1 = -inner(dot(sigma(u_mid, p_n), n), v)*ds(7)

# 2. Symmetric term (Stress tensor)
nitsche_2 = inner(v, dot(sigma(u_mid, p_n), n))*ds(7)

# 3. Tangent slip term (Navier slip condition)
u_t = u_mid - dot(u_mid, n)*n
v_t = v - dot(v, n)*n

nitsche_3 = beta * inner(u_t, v_t)*ds(7)                    # slip punishment
nitsche_4 = -nu * inner(dot(grad(u_mid), n), v_t)*ds(7)     # gradient term
nitsche_5 = -nu * inner(dot(grad(v), n), u_t)*ds(7)         # symmetric gradient term

# 4. Punishment for no penetration
nitsche_6 = alpha/h * dot(u_mid, n)*dot(v, n)*ds(7)

# Combine all Nitsche term
F_nitsche_slip = (
    nitsche_1 + nitsche_2 +
    nitsche_3 +# nitsche_4 + nitsche_5 +
    nitsche_6
)

In [10]:
theta = 0.5  # (Crank-Nicolson)

# Stabilization parameter
h = CellDiameter(mesh)
u_mag = sqrt(dot(u1, u1))
d1 = 1.0 / sqrt(1.0/dt**2 + (u_mag/h)**2 + 1e-10)
d2 = h * u_mag

u_mid = theta*u + (1 - theta)*u_n
u_mid1 = theta*u1 + (1 - theta)*u_n

# SUPG stabilization
F_SUPG = d1 * inner(
    (u - u_n)/dt + dot(grad(u_mid), u_mid1) + grad(p_n) - nu*div(grad(u_mid)),
    dot(grad(v), u_mid1)
) * dx \
         + d2 * div(u_mid) * div(v) * dx

# PSPG stabilization
F_PSPG = d1 * inner(div(u_mid1), q) * dx

# Momentum equation
Fu = (
    inner((u - u_n)/dt, v)*dx
    + inner(dot(grad(u_mid), u_mid1), v)*dx
    - p_n*div(v)*dx
#    + nu*inner(grad(u_mid), grad(v))*dx
    + F_SUPG
    + F_nitsche_slip  # Navier slip condition
)

# Pressure possion equation
Fp = (
    (inner(grad(p), grad(q))
    - inner(grad(p_n), grad(q)))*dx + F_PSPG
)

# Biliear form
auv1 = lhs(Fu)
Lv1 = rhs(Fu)
auv2 = lhs(Fp)
Lv2 = rhs(Fp)

print(dt)

0.02209912045588785


# 7. Computation

For the diffusion equation:

$$
\frac{\partial {u}}{\partial t} - \nu\nabla^2 u = 0
$$

the diffusive speed is $x \sim \sqrt{\nu T}$. Therefore, we need rather large $\nu$ to make transport clear.

In [ ]:
#T = 50*D   
T = 50*D
num_steps = int(T/dt)
# num_steps = int(300)
freq_pic = int(50)
hmesh = mesh.hmin()

u_n.assign(u0)
p_n.assign(p0)
ulist = [u0]
plist = [p0]
dtlist = [dt]

u_old = Function(V)

num_nnlin_iter = 5
prec = "amg" if has_krylov_solver_preconditioner("amg") else "default"

for j in range(num_steps):
    
    for k in range(5):
        A1 = assemble(auv1)
        b1 = assemble(Lv1)
        [bc.apply(A1, b1) for bc in bcu]
        [bc.apply(u1.vector()) for bc in bcu]
        # solve(A1, u1.vector(), b1, "gmres", "ilu")
        solve(A1, u1.vector(), b1, "bicgstab", "default")
        # u1.rename("Velocity", "velocity field")

        A2 = assemble(auv2)
        b2 = assemble(Lv2)
        [bc.apply(A2, b2) for bc in bcp]
        [bc.apply(p1.vector()) for bc in bcp]
        # solve(A2, p1.vector(), b2, "gmres", "ilu")
        solve(A2, p1.vector(), b2, "bicgstab", "default")
        p_n.assign(p1)
        process_bar(j*5+k+1,5*num_steps)
      
        # if k > 0:
        #     residual = u1.vector() - u_old.vector()
        #     if residual.norm("l2") < 1e-5:
        #         break
        # u_old.assign(u1)
        # p1.rename("Pressure", "pressure field")
        # file_p << p1
        # file_u << u1

    if j % 1 == 0:
        ulist.append(Function(V))
        ulist[-1].assign(u1)
        plist.append(Function(Q))
        plist[-1].assign(p1)
    
    u_n.assign(u1)
    p_n.assign(p1)
#    dt = CFL*hmesh/np.max(np.abs(u_n.vector().get_local()))
    dtlist.append(dt)

    # if j % freq_pic == 0:
    #     plotInnerU(ulist, j)
    #     plt.savefig('./nu_{}_v0_{}_dh_{}/u/u_{}.png'.format(nu, v0, hmesh, j))
    #     plt.close(1)

    #     plotU(plist, j)
    #     plt.savefig('./nu_{}_v0_{}_dh_{}/p/p_{}.png'.format(nu, v0, hmesh, j))
    #     plt.close(1)

Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.


# 8. Postprocess

For a body surrounded by the fluid, the net force can be expressed as

$$
\mathbf{F} = \int_{\partial\Omega} (-p \mathbf{I} + \nu \nabla \mathbf{u}) \cdot \mathbf{n} \, ds
$$

Then the lift coefficient $ C_L $ and drag coefficient $ C_D $ are respectively

$$
C_L = \frac{F_L}{\frac{1}{2}\rho U^2A},\quad C_D = \frac{F_D}{\frac{1}{2}\rho U^2A}
$$

where $ F_L $ is the lift force, the vertical component of $ \mathbf{F} $; $ F_D $ is the drag force, the horizontal component of $ \mathbf{F} $; $ A $ is the reference area, calculated by diameter of cylinder times the length unit.

In [ ]:
D = 0.1          # Diameter of cylinder
L_span = H       # Span length
A = D * L_span   # Reference area

U = v0           # Inflow velocity
rho = 1.0        # Density

n = FacetNormal(mesh)

def compute_force_coefficients(u_func, p_func, nu_val, D, L_span, U, rho):

    sigma = -p_func*Identity(3) + nu_val*(grad(u_func) + grad(u_func).T)
    traction = dot(sigma, n)
    
    F_drag = assemble(traction[0] * ds(7))
    F_lift = assemble(traction[1] * ds(7))
    
    dynamic_pressure = 0.5 * rho * U**2
    Cd = F_drag / (dynamic_pressure * A)
    Cl = F_lift / (dynamic_pressure * A)
    
    return Cd, Cl


Cd_list = []
Cl_list = []

for i in range(1, len(ulist)):
    u_step = ulist[i]
    p_step = plist[i]
    
    Cd, Cl = compute_force_coefficients(u_step, p_step, nu, D, L_span, U, rho)
    
    Cd_list.append(Cd)
    Cl_list.append(Cl)

Cd_array = np.array(Cd_list)
Cl_array = np.array(Cl_list)

time_steps = np.arange(len(Cd_list))

dt = dtlist[0]
time = dt * (time_steps + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time, Cd_array, label='$C_D$', color='blue')
plt.xlabel('Time')
plt.ylabel('Drag Coefficient $C_D$')
plt.title('Drag Coefficient over Time')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(time, Cl_array, label='$C_L$', color='red')
plt.xlabel('Time')
plt.ylabel('Lift Coefficient $C_L$')
plt.title('Lift Coefficient over Time')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

start_avg = max(0, len(Cd_array) - 100)
Cd_avg = np.mean(Cd_array[start_avg:])
Cl_avg = np.mean(Cl_array[start_avg:])
Cl_rms = np.std(Cl_array[start_avg:])  # Lift coefficient pulsation value (RMS)

print(f"Average Drag Coefficient (last 100 steps): {Cd_avg:.4f}")
print(f"Average Lift Coefficient (last 100 steps): {Cl_avg:.4f}")
print(f"RMS Lift Coefficient (last 100 steps): {Cl_rms:.4f}")

In [ ]:
from scipy.signal import find_peaks

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time, Cd_array, label='$C_D$', color='blue')
plt.xlabel('Time')
plt.ylabel('Drag Coefficient $C_D$')
plt.title('Drag Coefficient over Time')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(time, Cl_array, label='$C_L$', color='red')
plt.xlabel('Time')
plt.ylabel('Lift Coefficient $C_L$')
plt.title('Lift Coefficient over Time')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('cd_cl_over_time.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

plt.figure(figsize=(10, 4))
start_idx = max(0, len(time) - 100)
plt.plot(time[start_idx:], Cl_array[start_idx:], label='$C_L$', color='red', linewidth=1.5)
plt.xlabel('Time')
plt.ylabel('Lift Coefficient $C_L$')
plt.title('Lift Coefficient (Last 100 Steps)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('cl_last_100_steps.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

start_avg = max(0, len(Cd_array) - 100)
Cd_avg = np.mean(Cd_array[start_avg:])
Cl_avg = np.mean(Cl_array[start_avg:])
Cl_rms = np.std(Cl_array[start_avg:])

print(f"Average Drag Coefficient (last 100 steps): {Cd_avg:.4f}")
print(f"Average Lift Coefficient (last 100 steps): {Cl_avg:.4f}")
print(f"RMS Lift Coefficient (last 100 steps): {Cl_rms:.4f}")

# Strouhal number: St = f * D / U

Cl_analysis = Cl_array[start_avg:]
time_analysis = time[start_avg:]

dt_analysis = time_analysis[1] - time_analysis[0] if len(time_analysis) > 1 else dt

# Compute spectrum using Fourier transformation (FFT)
N = len(Cl_analysis)
yf = np.fft.fft(Cl_analysis - Cl_avg)
xf = np.fft.fftfreq(N, dt_analysis)[:N//2]

# Compute power spectrum density (PSD)
psd = 2.0/N * np.abs(yf[0:N//2])

non_zero_idx = xf > 1e-6
xf_nonzero = xf[non_zero_idx]
psd_nonzero = psd[non_zero_idx]

peak_idx = np.argmax(psd_nonzero)
f_dominant = xf_nonzero[peak_idx]

# Compute Strouhal number
St = f_dominant * D / U

plt.figure(figsize=(8, 4))
plt.plot(xf_nonzero, psd_nonzero, color='purple')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('Power Spectrum of Lift Coefficient')
plt.grid(True)
plt.axvline(x=f_dominant, color='red', linestyle='--', label=f'Dominant f = {f_dominant:.3f} Hz')
plt.legend()
plt.tight_layout()
plt.savefig('cl_power_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f"Vortex shedding frequency (f): {f_dominant:.6f} Hz")
print(f"Strouhal number (St = f*D/U): {St:.6f}")

# 9. Plot

In [ ]:
image_folder_u = './nu_{}_v0_{}_dh_{}_beta_{}/u'.format(nu, v0, hmesh, beta)
image_folder_p = './nu_{}_v0_{}_dh_{}_beta_{}/p'.format(nu, v0, hmesh, beta)
os.makedirs(image_folder_u, exist_ok = True)
os.makedirs(image_folder_p, exist_ok = True)

pvd_file = File("./nu_{}_v0_{}_dh_{}_beta_{}/u/velocity_output.pvd".format(nu, v0, hmesh, beta))

for i, u in enumerate(ulist):
    t = i * dt
    u.rename("Velocity", "velocity field")
    pvd_file << (u, t)

pvd_file1 = File("./nu_{}_v0_{}_dh_{}_beta_{}/p/pressure_output.pvd".format(nu, v0, hmesh, beta))

for i, p in enumerate(plist):
    t = i * dt
    p.rename("Pressure", "pressure field")
    pvd_file1 << (p, t)